In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from sft import *
from datetime import datetime, timedelta
from interpolation import interp1d
from utils import *
from scipy.ndimage import gaussian_filter

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/synoptic_maps/*')))

flux = []
dates = []

for file in files:

    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    data = rebin(data, 2)
    data = np.nanmean(data, axis=-1)
    data = (data[1:] + data[:-1]) / 2
    data = np.nan_to_num(data)

    date = datetime.fromisoformat(header['T_OBS'][:-4].replace('.', '').replace('_', 'T'))

    flux.append(data)
    dates.append(date)

flux = np.array(flux)
dates = np.array(dates)

/tmp/ipykernel_119022/889444508.py:13: RuntimeWarning: Mean of empty slice
  data = np.nanmean(data, axis=-1)


In [3]:
dsine = 1 / 359.5
lati = np.arange(-1,1 + dsine / 2, dsine)
lati = np.arcsin(lati.clip(-1,1)) * 180 / np.pi
lat = (lati[1:] + lati[:-1]) / 2

In [4]:
plt.figure(figsize=(10,8))
plt.plot(lat, flux[17])
plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

In [5]:
R = 695e8
u = 1200
d = 500e10
lam = 45
dt = timedelta(days=1).total_seconds()

xi = lati * np.pi / 180 * R
vi = u * flow(lati, a=0.5)
li = 2 * np.pi * R * np.cos(lati  * np.pi / 180)

y = flux[0].copy()
Q = []
for t in np.arange(dates[0], dates[23], timedelta(days=1)):
    y_ = interp1d(flux, dates, t)
    y = advect(y, xi, vi, dt, li)
    y = diffuse(y, xi, d, dt, li)
    y[np.abs(lat) < lam] = y_[np.abs(lat) < lam]

    Q.append(y)

Q = np.array(Q)

In [6]:
plt.figure(figsize=(10,8))
plt.plot(lat, flux[0])
plt.plot(lat, flux[23])
plt.plot(lat, Q[-1])
plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [11]:
plt.figure(figsize=(10,8))
plt.imshow(Q.T, 'bwr', aspect='auto', origin='lower', vmin=-5, vmax=5)
plt.tight_layout()

In [10]:
plt.figure(figsize=(10,8))
plt.imshow(flux_.T, 'bwr', aspect='auto', origin='lower', vmin=-5, vmax=5)#, interpolation='bicubic')
plt.tight_layout()

In [36]:
plt.figure(figsize=(10,8))
plt.plot(lati, vi)
plt.tight_layout()